# cortrix-skills · OpenAI Function Calling demo

Expose Cortrix as OpenAI functions with `as_openai_functions(kit)` (gpt-4o / gpt-4o-mini
/ gpt-4-turbo / gpt-3.5-turbo).

> **Skeleton notebook.** Cells are the canonical usage flow. A real run needs a live
> cortrix-server + an OpenAI API key (function-calling round-trip) and is exercised during
> integration (integration), not in standalone development.

## Install

```bash
pip install cortrix-skills[openai]
```

In [ ]:
from cortrix_skills import CortrixToolKit
from cortrix_skills.adapters import as_openai_functions

kit = CortrixToolKit(
    base_url="https://cortrix.example.com",
    api_key="sk-cortrix-...",
)

tools = as_openai_functions(kit)
print(f"{len(tools)} OpenAI function definitions")
tools[0]  # {type: function, function: {name, description, parameters}}

In [ ]:
from openai import OpenAI
from cortrix_skills.adapters.openai import dispatch_openai_tool_call

client = OpenAI(api_key="sk-...")
messages = [{"role": "user", "content": "find last week's notes on the MCP design"}]

resp = client.chat.completions.create(model="gpt-4o-mini", tools=tools, messages=messages)
msg = resp.choices[0].message
messages.append(msg)

for call in msg.tool_calls or []:
    content = dispatch_openai_tool_call(kit, call)
    messages.append({"role": "tool", "tool_call_id": call.id, "content": content})

final = client.chat.completions.create(model="gpt-4o-mini", tools=tools, messages=messages)
print(final.choices[0].message.content)

## Errors

On a Cortrix error, `dispatch_openai_tool_call` returns the four GEN-Agent fields as the
tool message content (JSON) instead of raising, so the model can reason about the failure.